In [13]:
import warnings

warnings.filterwarnings("ignore")

import deepchem as dc
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import matplotlib.pyplot as plt
from scipy import sparse
import seaborn as sns
import mlflow
import os
from data_loader import *

In [14]:
import random
import numpy as np
import torch
import torch_geometric
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    np.random.seed(seed)
    random.seed(seed)
    torch_geometric.seed_everything(seed)

In [15]:


def train(args, filename=None):
    with mlflow.start_run() as run:
        run_id = run.info.run_id
        mlflow.log_params(vars(args))
        if filename:
            mlflow.log_artifact( filename, artifact_path='code')     
        if args.gpu >= 0:
            device = torch.device('cuda:%d' % args.gpu)
        else:
            device = torch.device('cpu')
            
        maes, mapes, mses = [], [], []
        
        for trial in range(args.num_trial):
            setup_seed(trial)
            if args.featurizer == 'MACCS':
                featurizer = dc.feat.MACCSKeysFingerprint()
            elif args.featurizer == 'ECFP6':
                featurizer = dc.feat.CircularFingerprint(size=1024, radius=6)
            elif args.featurizer == 'Mordred':
                featurizer = dc.feat.MordredDescriptors(ignore_3D=True)
            else:
                raise NotImplementedError       
              
            if args.dataset == 'HOPV':
                tasks, datasets, transformers = dc.molnet.load_hopv(featurizer=featurizer, splitter=None, transformers=[])
                target_task = 4
                index_dir = './data/HOPV'
                dataset = datasets[0]
                data_mask =  dataset.y[:, target_task] != 0
                if args.target_mode == 'single':
                    dataset = dc.data.DiskDataset.from_numpy(dataset.X[data_mask], dataset.y[data_mask, target_task], dataset.w[data_mask, target_task], dataset.ids[data_mask])
                elif args.target_mode == 'multi':
                    dataset = dc.data.DiskDataset.from_numpy(dataset.X[data_mask], dataset.y[data_mask], dataset.w[data_mask], dataset.ids[data_mask])
                else:
                    raise NotImplementedError
            elif args.dataset == 'PolymerFA':
                target_task = 0             
                dataset = PolymerFADataset()
                index_dir = './data/Polymer_FA/processed/'
                data_mask =  dataset.y[:, target_task] != 0   
                X = featurizer.featurize(dataset.smiles)
                if args.target_mode == 'single':
                    dataset = dc.data.DiskDataset.from_numpy(X, dataset.y.numpy()[:, target_task], None, dataset.smiles)
                elif args.target_mode == 'multi':
                    dataset = dc.data.DiskDataset.from_numpy(X, dataset.y.numpy(), None, dataset.smiles)
                else:
                    raise NotImplementedError   
            elif args.dataset == 'nNFA':
                target_task = 0   
                dataset = nNFADataset()      
                index_dir = './data/Polymer_NFA_n/processed/'
                data_mask =  dataset.y[:, target_task] != 0   
                X = featurizer.featurize(dataset.smiles)
                if args.target_mode == 'single':
                    dataset = dc.data.DiskDataset.from_numpy(X, dataset.y.numpy()[:, target_task], None, dataset.smiles)
                elif args.target_mode == 'multi':
                    dataset = dc.data.DiskDataset.from_numpy(X, dataset.y.numpy(), None, dataset.smiles)
                else:
                    raise NotImplementedError     
            elif args.dataset == 'pNFA':
                target_task = 0   
                dataset = pNFADataset()     
                index_dir = './data/Polymer_NFA_p/processed/' 
                data_mask =  dataset.y[:, target_task] != 0   
                X = featurizer.featurize(dataset.smiles)
                if args.target_mode == 'single':
                    dataset = dc.data.DiskDataset.from_numpy(X, dataset.y.numpy()[:, target_task], None, dataset.smiles)
                elif args.target_mode == 'multi':
                    dataset = dc.data.DiskDataset.from_numpy(X, dataset.y.numpy(), None, dataset.smiles)
                else:
                    raise NotImplementedError     
            elif args.dataset == 'CEPDB':
                target_task = 0 
                dataset = CEPDBDataset()
                index_dir = '/data/CEPDB/processed/'
                X = torch.load('/data/CEPDB/MACCS.pt') if args.featurizer == 'MACCS' else sparse.load_npz('/data/CEPDB/ECFP6.npz').toarray()
                y = dataset.y.numpy()
                if args.target_mode == 'single':
                    y = y[:, target_task]  
                ids = np.arange(len(dataset))
            else:
                raise NotImplementedError
            
            # Split dataset
            if args.splitter == 'random':
                splitter = dc.splits.RandomSplitter()
            elif args.splitter == 'scaffold':
                splitter = dc.splits.ScaffoldSplitter()
            else:
                raise NotImplementedError
            # Train: 60%, Valid: 20%, Test: 20%
            if os.path.exists(os.path.join(index_dir, f'train_index_{args.frac_train}.pt')):
                train_index, valid_index, test_index = torch.load(os.path.join(index_dir, f'train_index_{args.frac_train}.pt')), torch.load(os.path.join(index_dir, f'valid_index_{args.frac_train}.pt')), torch.load(os.path.join(index_dir, f'test_index_{args.frac_train}.pt'))   
            else:
                train_index, valid_index, test_index = splitter.split(dataset, frac_train=0.6, frac_valid=0.2, frac_test=0.2) 
            if args.dataset != 'CEPDB':
                train_dataset, valid_dataset, test_dataset = dataset.select(train_index),  dataset.select(valid_index), dataset.select(test_index)
            else:
                train_dataset = dc.data.DiskDataset.from_numpy(X[train_index], y[train_index], None, ids[train_index])
                valid_dataset = dc.data.DiskDataset.from_numpy(X[valid_index], y[valid_index], None, ids[valid_index])
                test_dataset = dc.data.DiskDataset.from_numpy(X[test_index], y[test_index], None, ids[test_index])
            # Normalize target values
            if args.normalize:
                transformer = dc.trans.NormalizationTransformer(transform_y=True, dataset=train_dataset)
                train_dataset, test_dataset = transformer.transform(train_dataset), transformer.transform(test_dataset)
            
            # Model initialization
            if args.model == 'RF':
                rf = RandomForestRegressor()
                model = dc.models.SklearnModel(model=rf, use_weights=False)
            else:
                raise NotImplementedError
            
            # Training&Testing
            model.fit(train_dataset)
            y_preds = model.predict(test_dataset) 
            y_true = test_dataset.y
            if args.normalize:
                y_true= transformer.untransform(y_true)
                y_preds = transformer.untransform(y_preds)
            if args.target_mode == 'multi':
                y_true = y_true[:, target_task]
                y_preds = y_preds[:, target_task]
            mae = mean_absolute_error(y_true, y_preds)
            mape = mean_absolute_percentage_error(y_true, y_preds)
            mse = mean_squared_error(y_true, y_preds)
            mlflow.log_metrics({'MAE': mae, 'MAPE': mape, 'MSE': mse}, step=trial)
            maes.append(mae)
            mapes.append(mape)
            mses.append(mse)

        # Log average results
        avg_val = np.mean(maes)
        std_val = np.std(maes)
        mlflow.log_metric(f'MAE_mean', avg_val)
        mlflow.log_metric(f'MAE_std', std_val)      
        
        avg_val = np.mean(mapes)
        std_val = np.std(mapes)
        mlflow.log_metric(f'MAPE_mean', avg_val)
        mlflow.log_metric(f'MAPE_std', std_val)
        
        avg_val = np.mean(mses)
        std_val = np.std(mses)
        mlflow.log_metric(f'MSE_mean', avg_val)
        mlflow.log_metric(f'MSE_std', std_val)
          
            

In [ ]:
%%capture
import warnings

warnings.filterwarnings("ignore")

import mlflow
import argparse
exp_name = 'ML_Baselines'
mlflow.set_tracking_uri('file:///data/leylazhang_proj/PCE/mlruns')
try:
    mlflow.create_experiment(exp_name)
except:
    print("Experiment has been created")
mlflow.set_experiment(exp_name)


parser = argparse.ArgumentParser()
parser.add_argument('-dataset', type=str, default='PolymerFA', choices=['NFAs_51K','NFAs_1.2K','HOPV', 'PolymerFA', 'nNFA', 'pNFA', 'CEPDB'])
parser.add_argument('-normalize', type=bool, default=False)
parser.add_argument('-frac_train', type=float, default=0.6)
parser.add_argument('-target_mode', type=str, default='multi')
parser.add_argument('-splitter', type=str, default='scaffold')
parser.add_argument('-featurizer', type=str, default='ECFP6', choices=['MACCS', 'ECFP6', 'Mordred'])
parser.add_argument('-num_trial', type=int, default=5)
parser.add_argument('-gpu', type=int, default=0)
parser.add_argument('-model', type=str, default='RF')

for ds in ['NFAs_51K','NFAs_1.2K','PolymerFA', 'nNFA', 'pNFA']:
    parser.set_defaults(dataset=ds)
    if ds == 'CEPDB':
        parser.set_defaults(num_trial=1)
    for featurizer in ['Mordred']:
        parser.set_defaults(featurizer=featurizer)
        for model in ['RF']:
            parser.set_defaults(model=model)
            for splitter in ['scaffold']:
                parser.set_defaults(splitter=splitter)
                for target_mode in ['single']:
                    parser.set_defaults(target_mode=target_mode)
                    args=[]
                    args = parser.parse_args(args=args)                
                    with torch.cuda.device(f'cuda:{args.gpu}'):
                        torch.cuda.empty_cache()
                    train(args, None)

ValueError: setting an array element with a sequence.